# Klasifikasi Ganjil-Genap Kendaraan dari Plat Nomor dengan Pengolahan Citra Digital

**Kelompok 7 Pengolahan Citra Digital**
Anggota:
- Naura Alifa Damayanti
- Andi Nabil Safaraz
- Muhammad Iqbal Hilmi
- Dania Hafiza  
- Jundan Saiful Haq

Sistem ini menerima foto jalan raya, mencari semua plat nomor di dalamnya, lalu memberi label **Ganjil** atau **Genap** untuk setiap plat dari digit terakhirnya. Semua tahap memakai pengolahan citra konvensional, tanpa deep learning:

1. **Akuisisi dan enhancement:** grayscale, CLAHE (perluasan histogram equalization), median filter.
2. **Lokalisasi ROI plat:** Sobel-X dan Canny, threshold Otsu, closing/opening, filter bentuk kontur, verifikasi kandidat, lalu non-maximum suppression. Satu citra bisa menghasilkan beberapa ROI.
3. **Segmentasi karakter:** Otsu, opening/closing, connected component labeling, lalu pemilihan digit terakhir.
4. **Representasi fitur:** HOG (berbasis tepi), dengan opsi tambahan LBP (tekstur).
5. **Klasifikasi:** KNN, SVM, dan XGBoost untuk digit 0–9, lalu `digit % 2` untuk ganjil/genap.

**Cara menjalankan:** di Kaggle, tambahkan dua dataset (*Indonesian License Plate Dataset* dan *Indonesian License Plate Recognition Dataset*) lewat *Add Input*, lalu *Run All*. Di Colab, ekstrak kedua dataset ke `/content/`. Path dicari otomatis.

## 0. Setup

In [ ]:
import os
import re
import glob
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog, local_binary_pattern
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")
print("OpenCV:", cv2.__version__)

Semua parameter ada di sini. Nilai awal lokalisasi diambil dari notebook versi pertama; `crop_top_ratio`, metode enhancement, dan `min_plate_score` dipilih ulang di bagian 3 memakai split val.

In [ ]:
CFG = {
    "seed": 42,
    # --- Enhancement ---
    "enhance_method": "clahe",        # "clahe" | "he" | "none"
    "clahe_clip": 2.0,
    "clahe_grid": (8, 8),
    "median_ksize": 3,
    # --- Lokalisasi: pembangkit kandidat ---
    "crop_top_ratio": 0.50,           # bagian atas frame dibuang (langit, gedung)
    "sobel_close_kernel": (17, 3),
    "canny_close_kernel": (15, 3),
    "open_kernel": (3, 3),
    "plate_ar_range": (2.2, 4.5),     # rasio lebar/tinggi plat
    "plate_area_range": (0.0004, 0.025),  # luas kandidat relatif terhadap luas citra
    "plate_min_extent": 0.40,
    "max_candidates": 30,
    # --- Lokalisasi: verifikasi kandidat ---
    "max_mean_saturation": 85.0,
    "min_contrast_gap": 50.0,
    "fg_ratio_range": (0.10, 0.65),
    "n_chars_range": (4, 9),
    "max_h_cv": 0.30,
    "max_y_spread": 0.14,
    # --- Lokalisasi: banyak ROI per citra ---
    "min_plate_score": 0.10,          # ROI dengan skor di bawah ini dibuang (dipilih ulang di val)
    "nms_iou": 0.30,                  # ROI yang tumpang tindih lebih dari ini dengan ROI berskor lebih tinggi dibuang
    "max_plates": 6,                  # batas jumlah ROI per citra
    # --- Segmentasi karakter ---
    "roi_height": 80,
    "use_morphology": True,
    "digit_size": (32, 32),
    # --- Fitur ---
    "use_lbp": False,
    # --- Evaluasi ---
    "iou_match": 0.30,                # ROI dianggap mengenai plat GT jika IoU >= ini
    "iou_digit": 0.50,                # digit terpilih dianggap benar jika IoU dengan bbox digit GT >= ini
}
np.random.seed(CFG["seed"])

In [ ]:
SEARCH_ROOTS = ["/kaggle/input", "/content", "."]


def find_dataset_root(marker: str):
    """Mencari folder dataset berdasarkan nama file/folder penanda di dalamnya."""
    for root in SEARCH_ROOTS:
        if not os.path.exists(root):
            continue
        hits = glob.glob(os.path.join(root, "**", marker), recursive=True)
        if hits:
            return str(Path(sorted(hits, key=len)[0]).parent)
    return None


MAIN_DIR = find_dataset_root("labelswithLP")      # Indonesian License Plate Dataset
REC_DIR = find_dataset_root("classes.names")      # Indonesian License Plate Recognition Dataset
assert MAIN_DIR is not None, "Indonesian License Plate Dataset tidak ditemukan. Tambahkan lewat Add Input."
assert REC_DIR is not None, "Indonesian License Plate Recognition Dataset tidak ditemukan. Tambahkan lewat Add Input."

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
SPLITS = ["train", "val", "test"]
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

print("MAIN_DIR:", MAIN_DIR)
print("REC_DIR :", REC_DIR)
print("OUT_DIR :", OUT_DIR)

## 1. Ground truth per plat (Indonesian License Plate Dataset)

Setiap baris di `labelswithLP` berisi satu plat: `kelas cx cy w h TEKS`. Satu gambar bisa berisi beberapa baris. Tabel `plates` menyimpan satu baris per plat, termasuk bbox. Bbox ini tidak masuk ke pipeline, melainkan hanya dipakai untuk mencocokkan hasil lokalisasi dengan plat yang benar dan untuk evaluasi.

In [ ]:
def read_yolo_rows(path: str):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 5:
                rows.append(parts)
    return rows


def last_digit_of_plate(text: str):
    """Digit terakhir dari kelompok angka plat Indonesia, misal B1184UJY -> 4."""
    cleaned = re.sub(r"[^A-Z0-9]", "", str(text).upper())
    match = re.match(r"^[A-Z]{1,2}(\d{1,4})[A-Z]{0,3}$", cleaned)
    if match:
        return int(match.group(1)[-1])
    digits = re.findall(r"\d", cleaned)
    return int(digits[-1]) if digits else None


def load_plates(split: str) -> pd.DataFrame:
    img_dir = os.path.join(MAIN_DIR, "images", split)
    lbl_dir = os.path.join(MAIN_DIR, "labelswithLP", split)
    records = []
    for img_name in sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)):
        stem = os.path.splitext(img_name)[0]
        rows = read_yolo_rows(os.path.join(lbl_dir, stem + ".txt"))
        for k, parts in enumerate(rows, start=1):
            text = "".join(parts[5:]).upper()
            digit = last_digit_of_plate(text) if text else None
            records.append({
                "split": split,
                "image_name": img_name,
                "image_path": os.path.join(img_dir, img_name),
                "stem": stem,
                "plate_idx": k,
                "cx": float(parts[1]), "cy": float(parts[2]),
                "bw": float(parts[3]), "bh": float(parts[4]),
                "plate_text": text,
                "last_digit": digit,
                "parity": None if digit is None else digit % 2,
            })
    return pd.DataFrame(records)


plates = pd.concat([load_plates(s) for s in SPLITS], ignore_index=True)
images = (
    plates.groupby(["split", "image_name", "image_path"], as_index=False)
    .agg(n_plates=("plate_idx", "size"))
)

print("Jumlah plat per split :", plates.groupby("split").size().to_dict())
print("Jumlah gambar per split:", images.groupby("split").size().to_dict())
print("Plat tanpa digit terbaca:", int(plates["last_digit"].isna().sum()))
print("\nJumlah plat per gambar (berapa gambar):")
display(images.pivot_table(index="n_plates", columns="split", values="image_name", aggfunc="count").fillna(0).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_plates = plates[(plates["split"] == "train") & plates["last_digit"].notna()]
sns.countplot(x=train_plates["last_digit"].astype(int), color="#4C72B0", ax=axes[0])
axes[0].set_title("Distribusi digit terakhir (plat train)")
axes[0].set_xlabel("Digit terakhir")
sns.countplot(x=images[images["split"] == "train"]["n_plates"], color="#55A868", ax=axes[1])
axes[1].set_title("Jumlah plat per gambar (train)")
axes[1].set_xlabel("Plat dalam satu gambar")
plt.tight_layout()
plt.show()

## 2. Data karakter (Indonesian License Plate Recognition Dataset)

Dataset ini berisi crop plat (`train001_1.jpg`, `train002_1.jpg`, `train002_2.jpg`, ...) dengan bbox per karakter dan kelas `0–9, A–Z`. Nama file menunjukkan crop ke-k dari gambar di dataset utama, dan pembagian train/val/test-nya sama. Karena itu, classifier digit dilatih dari crop train saja, sehingga gambar test tidak pernah terlihat saat pelatihan.

Sel berikut juga mengecek apakah teks hasil menyusun karakter dari kiri ke kanan sama dengan teks plat di dataset utama.

In [ ]:
with open(os.path.join(REC_DIR, "classes.names"), "r", encoding="utf-8") as f:
    CLASSES = [ln.strip() for ln in f if ln.strip()]
print("Kelas:", "".join(CLASSES))


def load_recognition(split: str) -> pd.DataFrame:
    img_dir = os.path.join(REC_DIR, "images", split)
    lbl_dir = os.path.join(REC_DIR, "labels", split)
    records = []
    for img_name in sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)):
        stem = os.path.splitext(img_name)[0]
        m = re.match(r"^(.*)_(\d+)$", stem)
        src_stem, plate_idx = (m.group(1), int(m.group(2))) if m else (stem, 1)
        chars = []
        for parts in read_yolo_rows(os.path.join(lbl_dir, stem + ".txt")):
            cls = int(parts[0])
            chars.append((cls, float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))
        chars.sort(key=lambda c: c[1])  # kiri ke kanan
        records.append({
            "split": split,
            "crop_path": os.path.join(img_dir, img_name),
            "stem": src_stem,
            "plate_idx": plate_idx,
            "rec_text": "".join(CLASSES[c[0]] for c in chars),
            "chars": chars,
        })
    return pd.DataFrame(records)


rec = pd.concat([load_recognition(s) for s in SPLITS], ignore_index=True)
print("Jumlah crop per split:", rec.groupby("split").size().to_dict())

# Cek kecocokan dengan dataset utama
check = rec.merge(plates[["split", "stem", "plate_idx", "plate_text"]], on=["split", "stem", "plate_idx"], how="left")
same_idx = (check["rec_text"] == check["plate_text"]).mean()
texts_per_image = plates.groupby(["split", "stem"])["plate_text"].apply(set).to_dict()
same_image = np.mean([
    r["rec_text"] in texts_per_image.get((r["split"], r["stem"]), set()) for _, r in rec.iterrows()
])
print(f"Teks crop == teks plat dengan indeks sama : {same_idx:.1%}")
print(f"Teks crop ada di gambar asal yang sama     : {same_image:.1%}")

## 4. Segmentasi Karakter dan Pemilihan Digit Terakhir

1. ROI disamakan ukurannya, diubah ke grayscale, lalu diperjelas dan dikurangi noise.
2. Otsu digunakan untuk memisahkan karakter dari latar belakang.
3. Opening dan closing membantu menghilangkan bintik dan menyambungkan karakter.
4. CCL mendeteksi dan menyaring komponen karakter.
5. Karakter dikelompokkan, lalu digit paling kanan pada kelompok angka dipilih.
6. Karakter dipotong, diberi padding, dan di-resize menjadi 32×32 piksel.

> Add blockquote



In [ ]:
def preprocess_plate_roi(roi_bgr: np.ndarray, cfg=CFG):
    if roi_bgr is None or roi_bgr.size == 0:
        return None
    h, w = roi_bgr.shape[:2]
    if h < 5 or w < 10:
        return None
    target_h = cfg["roi_height"]
    new_w = max(20, int(round(w * target_h / float(h))))
    roi = cv2.resize(roi_bgr, (new_w, target_h), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    enh = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gray)
    blur = cv2.medianBlur(enh, 3)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    border = np.concatenate([otsu[0, :], otsu[-1, :], otsu[:, 0], otsu[:, -1]])
    if border.mean() > 127:
        otsu = cv2.bitwise_not(otsu)
    cleaned = otsu
    if cfg["use_morphology"]:
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        cleaned = cv2.morphologyEx(cv2.morphologyEx(otsu, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    return {"roi_resized": roi, "gray": gray, "enhanced": enh, "otsu": otsu, "cleaned": cleaned,
            "scale_x": new_w / float(w), "scale_y": target_h / float(h)}


def crop_to_patch(binary: np.ndarray, box, size, pad: int = 2) -> np.ndarray:
    """Potong kotak (x, y, w, h) dari citra biner, jadikan persegi, resize ke `size`."""
    x, y, w, h = box
    H, W = binary.shape[:2]
    x1, y1, x2, y2 = max(0, x - pad), max(0, y - pad), min(W, x + w + pad), min(H, y + h + pad)
    crop = binary[y1:y2, x1:x2]
    side = max(crop.shape[:2])
    square = np.zeros((side, side), dtype=np.uint8)
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    square[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    return cv2.resize(square, size, interpolation=cv2.INTER_AREA)


def character_boxes(binary: np.ndarray):
    h_roi, w_roi = binary.shape[:2]
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        ar = w / float(h) if h > 0 else 0
        if not (0.28 <= h / h_roi <= 0.85 and 0.02 <= w / w_roi <= 0.25 and 0.12 <= ar <= 1.15 and area >= 25):
            continue
        if (y + h / 2.0) > 0.78 * h_roi and h / h_roi < 0.38:   # baris kecil masa berlaku di bawah plat
            continue
        boxes.append((int(x), int(y), int(w), int(h)))
    return sorted(boxes, key=lambda b: b[0])


def select_last_digit(boxes):
    """Pilih komponen paling kanan pada kelompok angka."""
    if not boxes:
        return None
    if len(boxes) < 3:
        return boxes[-1]
    gaps = [boxes[i + 1][0] - (boxes[i][0] + boxes[i][2]) for i in range(len(boxes) - 1)]
    gap_thresh = max(0.35 * np.median([b[2] for b in boxes]), np.percentile(gaps, 70))
    groups = [[boxes[0]]]
    for i, gap in enumerate(gaps):
        if gap > gap_thresh:
            groups.append([boxes[i + 1]])
        else:
            groups[-1].append(boxes[i + 1])
    if len(groups) >= 3:
        return max(groups[1:-1], key=len)[-1]
    if len(groups) == 2:
        return groups[1][-1] if len(groups[0]) <= 2 else groups[0][-1]
    return boxes[int(round(0.65 * (len(boxes) - 1)))]


def segment_last_digit(roi_bgr: np.ndarray, cfg=CFG):
    """Mengembalikan (patch 32x32 atau None, info debug)."""
    prep = preprocess_plate_roi(roi_bgr, cfg)
    if prep is None:
        return None, None
    boxes = character_boxes(prep["cleaned"])
    chosen = select_last_digit(boxes)
    prep["char_boxes"], prep["selected_box"] = boxes, chosen
    if chosen is None:
        return None, prep
    return crop_to_patch(prep["cleaned"], chosen, cfg["digit_size"]), prep

In [ ]:
# Contoh segmentasi pada crop plat dari Recognition Dataset (train)
fig, axes = plt.subplots(3, 4, figsize=(16, 7))
for row_i, (_, r) in enumerate(rec[rec["split"] == "train"].head(3).iterrows()):
    crop = cv2.imread(r["crop_path"])
    patch, dbg = segment_last_digit(crop)
    vis = dbg["roi_resized"].copy()
    for (x, y, w, h) in dbg["char_boxes"]:
        cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 200, 0), 1)
    if dbg["selected_box"] is not None:
        x, y, w, h = dbg["selected_box"]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 0, 255), 2)
    axes[row_i, 0].imshow(cv2.cvtColor(dbg["roi_resized"], cv2.COLOR_BGR2RGB)); axes[row_i, 0].set_title(f"ROI | GT {r['rec_text']}")
    axes[row_i, 1].imshow(dbg["otsu"], cmap="gray"); axes[row_i, 1].set_title("Otsu")
    axes[row_i, 2].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[row_i, 2].set_title("CCL (merah = digit terakhir)")
    if patch is not None:
        axes[row_i, 3].imshow(patch, cmap="gray")
    axes[row_i, 3].set_title("Patch 32x32")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()